In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [3]:
data = pd.read_stata(r"Z:\survey\ECU\ENEMDU\1991\m11\data_orig\ECU_1991m11.dta") # para bases de stata
#data = pd.read_stata(r"datos/ECU_1991m11_BID.dta") # para bases de stata

## Revisar los datos

- rn - región natural
- estrato - estrato
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingobr - Ingresos como obrero o empleado
- ingpat - Ingresos como patrono o cuenta propia
- ingalq - Ingresos por alquileres, rentas o interese
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- fexp - factor de expansión
- ingrl - ingresos

El valor de 'ingobr' es el ingreso laboral monetario, no hay datos sobre ingreso laboral no monetario, las otras variables son ingreso no laboral monetario y no monetario e ingrl es un ingreso total

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no ocupado, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba ocupado tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39071 entries, 0 to 39070
Data columns (total 82 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   estrato   39071 non-null  int8    
 1   rn        39071 non-null  int8    
 2   ciudad    39071 non-null  object  
 3   zona      39071 non-null  object  
 4   sector    39071 non-null  object  
 5   vivienda  39071 non-null  object  
 6   hogar     39071 non-null  object  
 7   formul    39071 non-null  object  
 8   numpers   39071 non-null  int8    
 9   persona   39071 non-null  int8    
 10  resultad  39071 non-null  category
 11  reljefe   39071 non-null  category
 12  edad      39071 non-null  category
 13  sexo      39071 non-null  category
 14  nivinst   33810 non-null  category
 15  anoinst   31528 non-null  float64 
 16  asistea   32532 non-null  category
 17  sabele    29907 non-null  category
 18  iess      29907 non-null  category
 19  lininf    39071 non-null  object  
 20  donnac

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [5]:
data.columns

Index(['estrato', 'rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar',
       'formul', 'numpers', 'persona', 'resultad', 'reljefe', 'edad', 'sexo',
       'nivinst', 'anoinst', 'asistea', 'sabele', 'iess', 'lininf', 'donnac',
       'lugnac', 'siemvic', 'donvian', 'lugvian', 'cuanvic', 'trabajo',
       'actayuda', 'hortrasa', 'ratmeh', 'ratmeh1', 'ratmah', 'aunotra',
       'pornot', 'bustrasa', 'bustrama', 'amigos', 'directo', 'prensa',
       'agepu', 'agepri', 'tresne', 'tiembus', 'motnobus', 'deseatra',
       'condina', 'trabant', 'tiemnot', 'rama', 'grupo', 'catetrab',
       'pertrabn', 'numtrab', 'hortrahp', 'hortrahs', 'hortraho', 'ramas',
       'grupos', 'cates', 'ingobr', 'ingpat', 'ingalq', 'ingjub', 'ingotr',
       'oct', 'sep', 'ago', 'jul', 'jun', 'may', 'abr', 'mar', 'feb', 'ene',
       'dic', 'nov', 'condact', 'secins', 'subequ', 'ingrl', 'peamsiu',
       'fexp'],
      dtype='object')

In [6]:
data = data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'persona', 'numpers', 'edad', 'ingobr', 'ingpat', 'ingalq',
      'ingjub', 'ingotr', 'fexp', 'ingrl', 'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']]

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingobr' siempre que reportan estar ocupados en un mes

In [7]:
data['ingr_ene'] = data.apply(lambda x: x['ingobr'] if x['ene'] == 'ocupado' else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingobr'] if x['feb'] == 'ocupado' else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingobr'] if x['mar'] == 'ocupado' else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingobr'] if x['abr'] == 'ocupado' else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingobr'] if x['may'] == 'ocupado' else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingobr'] if x['jun'] == 'ocupado' else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingobr'] if x['jul'] == 'ocupado' else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingobr'] if x['ago'] == 'ocupado' else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingobr'] if x['sep'] == 'ocupado' else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingobr'] if x['oct'] == 'ocupado' else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingobr'] if x['nov'] == 'ocupado' else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingobr'] if x['dic'] == 'ocupado' else None, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014, utilizamos el IPC de Estados Unidos para ajustar por inflación ya que no podemos usar la inflación en sucres si queremos dejar el valor final en dólares de 2014

In [8]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 1991]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc y tipo de cambio

In [9]:
ipc_dict = dict(zip(datos_actual['trimestre'], datos_actual['IPC Estados Unidos']))

ipc_base_dict = dict(zip(datos_base['trimestre'], datos_base['IPC Estados Unidos']))

tipo_cambio_dict = dict(zip(datos_actual['trimestre'], datos_actual['tipo de cambio']))

### Asignamos el ipc y tipo de cambio correspondiente según trimestre

$\begin{equation}
    ingr_{USD-base-2014}^{i} = \frac{ingr_{sucres}^{i}}{tipo-de-cambio^{i}}\left( \frac{ipcUSA^{i}_{2014}}{ipcUSA^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

In [10]:
# Función que asigna valores correspondientes
def asigna_ipc(trimestre):
    return ipc_dict.get(trimestre, {})

def asigna_ipc_base(trimestre):
    return ipc_base_dict.get(trimestre, {})

In [11]:
data['ipc_t1'] = ipc_dict.get(1)
data['ipc_base_t1'] = ipc_base_dict.get(1)
data['tipo_cambio_t1'] = tipo_cambio_dict.get(1)

data['ipc_t2'] = ipc_dict.get(2)
data['ipc_base_t2'] = ipc_base_dict.get(2)
data['tipo_cambio_t2'] = tipo_cambio_dict.get(2)

data['ipc_t3'] = ipc_dict.get(3)
data['ipc_base_t3'] = ipc_base_dict.get(3)
data['tipo_cambio_t3'] = tipo_cambio_dict.get(3)

data['ipc_t4'] = ipc_dict.get(4)
data['ipc_base_t4'] = ipc_base_dict.get(4)
data['tipo_cambio_t4'] = tipo_cambio_dict.get(4)

In [12]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [13]:
# Ingreso real por mes
data['ingr_ene_r'] = (data['ingr_ene'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_feb_r'] = (data['ingr_feb'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_mar_r'] = (data['ingr_mar'] / data['tipo_cambio_t1']) * data['def_t1']
data['ingr_abr_r'] = (data['ingr_abr'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_may_r'] = (data['ingr_may'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jun_r'] = (data['ingr_jun'] / data['tipo_cambio_t2']) * data['def_t2']
data['ingr_jul_r'] = (data['ingr_jul'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_ago_r'] = (data['ingr_ago'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_sep_r'] = (data['ingr_sep'] / data['tipo_cambio_t3']) * data['def_t3']
data['ingr_oct_r'] = (data['ingr_oct'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_nov_r'] = (data['ingr_nov'] / data['tipo_cambio_t4']) * data['def_t4']
data['ingr_dic_r'] = (data['ingr_dic'] / data['tipo_cambio_t4']) * data['def_t4']

Ingreso mensual promedio en el trimeste

In [14]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

## Regiones

In [15]:
print(data['ciudad'][0])
fac = data['ciudad'].apply(lambda x: len(str(x))).min()
fac

010150


6

In [ ]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == (fac - 1) else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

In [18]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [19]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data['region'] = data['ciudad_2'].map(codigo_region)

In [20]:
data['region'].value_counts()

region
Guayas                  10309
Pichincha                6908
Sierra                   5675
El Oro                   3453
Azuay                    3333
Amazonía                 3118
Manabí                   2954
Los Ríos                 2142
Costa, Santo Domingo     1179
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [21]:
columnas_idef = ['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

8243

In [22]:
data[['rn', 'estrato', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'idef_hogar', 'persona', 'numpers']]

,rn,estrato,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,1,3,010150,001,005,01,1,13010150001005011,1,3
1,1,3,010150,001,005,01,1,13010150001005011,2,3
2,1,3,010150,001,005,01,1,13010150001005011,3,3
3,1,3,010150,001,005,02,1,13010150001005021,1,2
4,1,3,010150,001,005,02,1,13010150001005021,2,2
...,...,...,...,...,...,...,...,...,...,...
39066,3,0,210450,001,011,11,1,30210450001011111,5,5
39067,3,0,210450,001,011,12,1,30210450001011121,1,4
39068,3,0,210450,001,011,12,1,30210450001011121,2,4
39069,3,0,210450,001,011,12,1,30210450001011121,3,4


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [23]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [24]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [25]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h    295.581289
ingr_t2_h    270.916819
ingr_t3_h    264.935781
ingr_t4_h    232.769974
dtype: object

## Sacamos edades negativas y mayores a 100 años

In [26]:
len(data)

39071

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [27]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [28]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

39071

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [29]:
k = 0.4
s = 0.9

In [30]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [31]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [32]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [33]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']]

,ingr_t_t1,ingr_t_t2,ingr_t_t3,ingr_t_t4
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...
39066,9.942692,9.093862,8.851951,7.812702
39067,54.824632,50.144132,48.810219,43.079731
39068,54.824632,50.144132,48.810219,43.079731
39069,54.824632,50.144132,48.810219,43.079731


In [34]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  60.072161532516716
Mediana del ingreso individual descontando cargas familiares t4:  41.74446318843047


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [35]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 1991

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [36]:
resultados_list = []

# Para cada trimeste
for t in [1, 2, 3, 4]:
    salario = salario_dict.get(t)
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    
    # Agrupa por región
    grouped = data.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / mediana            
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,1991,1,Amazonía,0.532346,0.229078,0.129762,0.099239,0.181941,0.252486,86.158260,61.985935,36.0,0.580777
1,1991,1,Azuay,0.509955,0.208090,0.117609,0.107014,0.198263,0.278131,103.542330,63.297947,36.0,0.568739
2,1991,1,"Costa, Santo Domingo",0.695879,0.356204,0.214162,0.093033,0.172114,0.240394,61.699741,42.190129,36.0,0.853280
3,1991,1,El Oro,0.569748,0.260460,0.155944,0.076570,0.146453,0.211344,72.209435,57.476773,36.0,0.626340
4,1991,1,Guayas,0.639555,0.281143,0.165193,0.083543,0.159797,0.229756,72.140308,52.731005,36.0,0.682710
5,1991,1,Los Ríos,0.606590,0.260439,0.148094,0.064220,0.126144,0.186510,69.487052,56.919993,36.0,0.632467
6,1991,1,Manabí,0.624227,0.309396,0.194585,0.087913,0.169542,0.245846,68.773847,51.438929,36.0,0.699859
7,1991,1,Pichincha,0.548518,0.246860,0.143286,0.110171,0.206102,0.289951,97.465492,58.681986,36.0,0.613476
8,1991,1,Sierra,0.695752,0.341957,0.212412,0.088005,0.168483,0.242621,62.609050,45.535994,36.0,0.790583
9,1991,2,Amazonía,0.622878,0.257711,0.147937,0.099048,0.181670,0.252204,79.078395,56.724749,36.0,0.634644


### Inserta los cálculos en la base final

In [37]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [38]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')